# EdgeRetina | Pré-processamento de imagens do APTOS 2019

Notebook para Google Colab, adaptado ao arquivo `APTOS.zip` já baixado no computador. O objetivo é reproduzir o pré-processamento descrito por Guidoum et al. (2026), sem treinar uma rede neural.

**Execute as células na ordem.** Não é necessário ter acesso ao Kaggle nem uma GPU. O arquivo ZIP não está incluído neste notebook.

## 1. Metodologia do artigo

A seção 2, a Figura 1 e a Tabela 1 do artigo EdgeRetina (páginas 236 a 238) definem as etapas abaixo.

| Etapa | Configuração |
|---|---|
| Redimensionamento | 128 × 128 pixels, três canais RGB |
| Normalização | Intensidades de 0 a 1 |
| Inversão horizontal | Aleatória, somente em treino |
| Contraste | Fator aleatório entre 0,90 e 1,10, somente em treino |
| Rótulo binário | Graus 0 e 1 → 0; graus 2, 3 e 4 → 1 |
| Lotes | 32 imagens |

O artigo informa 15% de validação. Nesta adaptação, preservamos os conjuntos fornecidos em seu ZIP, sem gerar uma nova divisão. Os percentuais reais serão calculados. O artigo não informa a interpolação do redimensionamento ou a probabilidade de inversão; adotamos INTER_AREA para redução, INTER_LINEAR para ampliação e probabilidade de 50%. Não aplicamos CLAHE, recorte, padding, segmentação ou filtros, pois não são etapas do método principal descrito no EdgeRetina.

## 2. Como enviar seu arquivo ZIP

1. Abra [Google Colab](https://colab.research.google.com/) e faça o upload deste notebook em **Arquivo > Fazer upload de notebook**.
2. Execute as duas próximas células. A segunda abrirá o seletor de arquivos. Escolha o arquivo `C:\Users\aleff\Downloads\PIBIC\datasets\APTOS.zip`.
3. Aguarde a conclusão da transferência antes de executar as demais células. O Colab não acessa diretamente a pasta do Windows.

Para arquivos grandes, prefira enviar o ZIP ao Google Drive, por exemplo para `Meu Drive/APTOS.zip`, e configurar `ORIGEM = "drive"`. Assim, você não precisa reenviar o ZIP a cada sessão. Outra opção é enviar o ZIP pelo painel **Arquivos** do Colab e escolher `ORIGEM = "local"`.

Estrutura esperada, conforme a imagem enviada. O Windows pode ocultar a extensão `.csv`:

```text
APTOS.zip
├── train_images/
├── val_images/
├── test_images/
├── train_1.csv
├── valid.csv
└── test.csv
```

O ZIP pode conter uma pasta externa. O código localizará os seis itens mesmo que estejam dentro dela.

In [ ]:
from pathlib import Path, PurePosixPath
import os
import shutil
import tempfile
import zipfile
import warnings
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

SEMENTE = 42
np.random.seed(SEMENTE)
cv2.setNumThreads(1)

# Escolha "upload", "drive" ou "local".
ORIGEM = "upload"
ZIP_DRIVE = "/content/drive/MyDrive/APTOS.zip"
ZIP_LOCAL = "/content/APTOS.zip"
TRABALHO = Path("/content/aptos_edge")
TRABALHO.mkdir(parents=True, exist_ok=True)

# Deixe None para detectar as colunas automaticamente.
# Se o CSV usar outro nome, informe a coluna exata, por exemplo "image_id".
COLUNA_ID = None
COLUNA_ROTULO = None
# False ignora os rótulos do teste, caso sejam ausentes ou fictícios.
TESTE_ROTULADO = None  # None = detectar; True = exigir; False = ignorar

print("Python:", sys.version.split()[0])
print("OpenCV:", cv2.__version__)
print("Origem:", ORIGEM)

In [ ]:
if ORIGEM == "upload":
    from google.colab import files
    # Reaproveita o ZIP se já estiver no ambiente desta sessão.
    ZIP = Path(ZIP_LOCAL)
    if not ZIP.is_file():
        print("Selecione APTOS.zip no seu computador.")
        enviados = files.upload()
        nomes = [nome for nome in enviados if nome.lower().endswith(".zip")]
        del enviados
        if len(nomes) != 1:
            raise ValueError("Envie exatamente um arquivo ZIP, o APTOS.zip.")
        ZIP = Path(nomes[0]).resolve()
elif ORIGEM == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP = Path(ZIP_DRIVE)
elif ORIGEM == "local":
    ZIP = Path(ZIP_LOCAL)
else:
    raise ValueError("ORIGEM deve ser upload, drive ou local.")

if not ZIP.is_file() or not zipfile.is_zipfile(ZIP):
    raise ValueError(f"Arquivo ZIP ausente ou inválido: {ZIP}")
print(f"ZIP: {ZIP}")
print(f"Tamanho: {ZIP.stat().st_size / 1024**2:.1f} MiB")

## 3. Extrair e localizar os conjuntos

O código extrai o ZIP em um diretório temporário e procura os pares CSV/pasta de imagens. Não altera o arquivo ZIP original. Se faltar algum item ou houver mais de uma pasta candidata, ele exibe um erro para evitar misturar conjuntos.

In [ ]:
DIVISOES = {
    "treino": ("train_1.csv", "train_images"),
    "validacao": ("valid.csv", "val_images"),
    "teste": ("test.csv", "test_images"),
}

def extrair_zip(arquivo, destino):
    with zipfile.ZipFile(arquivo) as z:
        itens = z.infolist()
        necessario = sum(i.file_size for i in itens)
        if necessario > shutil.disk_usage(destino).free:
            raise OSError("Espaço insuficiente no Colab para extrair o ZIP.")
        for item in itens:
            p = PurePosixPath(item.filename.replace("\\", "/"))
            if p.is_absolute() or ".." in p.parts or any(":" in s for s in p.parts):
                raise ValueError(f"Caminho inválido no ZIP: {item.filename}")
            alvo = destino.joinpath(*p.parts)
            if item.is_dir():
                alvo.mkdir(parents=True, exist_ok=True)
            else:
                alvo.parent.mkdir(parents=True, exist_ok=True)
                with z.open(item) as entrada, alvo.open("wb") as saida:
                    shutil.copyfileobj(entrada, saida, length=1024**2)

def localizar_unico(base, nome, pasta=False):
    candidatos = [p for p in base.rglob("*") if p.name.lower() == nome.lower()
                  and (p.is_dir() if pasta else p.is_file())
                  and "__MACOSX" not in p.parts]
    if len(candidatos) != 1:
        raise ValueError(f"Esperado um item {nome}; encontrados: {candidatos}")
    return candidatos[0]

# Pasta nova para impedir que arquivos de outra extração se misturem.
DATASET_DIR = Path(tempfile.mkdtemp(prefix="extraido_", dir=TRABALHO))
extrair_zip(ZIP, DATASET_DIR)
CAMINHOS = {}
for nome, (csv, pasta) in DIVISOES.items():
    CAMINHOS[nome] = (localizar_unico(DATASET_DIR, csv),
                      localizar_unico(DATASET_DIR, pasta, pasta=True))
    print(nome, "|", CAMINHOS[nome][0].name, "|", CAMINHOS[nome][1].name)
print("Extraído em:", DATASET_DIR)

## 4. Conferir os CSVs

Os nomes das colunas não estavam visíveis na captura de tela. O APTOS original usa `id_code` e `diagnosis`. A próxima célula mostra as colunas e alguns registros de cada CSV, antes de tentar associá-los às imagens. São aceitos alguns nomes alternativos comuns. Se necessário, ajuste `COLUNA_ID` e `COLUNA_ROTULO` na configuração e execute as células novamente.

Os graus originais são 0 (sem RD), 1 (leve), 2 (moderada), 3 (grave) e 4 (proliferativa). O teste pode não ter rótulos verdadeiros; nesse caso ele será usado apenas para inferência futura.

In [ ]:
TABELAS_BRUTAS = {}
for nome, (csv, pasta) in CAMINHOS.items():
    # Detecta vírgula ou ponto e vírgula; preserva identificadores como texto.
    with csv.open(encoding="utf-8-sig") as f:
        cabecalho = f.readline()
    separador = ";" if ";" in cabecalho and "," not in cabecalho else ("\t" if "\t" in cabecalho and "," not in cabecalho else ",")
    df = pd.read_csv(csv, sep=separador, dtype=str, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    TABELAS_BRUTAS[nome] = df
    print(f"\n{nome}: {len(df)} registros | colunas: {df.columns.tolist()}")
    display(df.head(3))

In [ ]:
EXTENSOES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
IDS = ("id_code", "image_id", "id", "image", "images", "filename", "file_name", "arquivo", "path", "image_path")
ROTULOS = ("diagnosis", "label", "labels", "grade", "level", "class", "target", "codigo_diagnostico", "severity")

def escolher_coluna(df, preferida, alternativas, obrigatoria=True):
    mapa = {str(col).strip().lower(): col for col in df.columns}
    if preferida is not None:
        if preferida.strip().lower() not in mapa:
            if obrigatoria:
                raise ValueError(f"Coluna {preferida} ausente: {df.columns.tolist()}")
            return None
        return mapa[preferida.strip().lower()]
    for opcao in alternativas:
        if opcao in mapa:
            return mapa[opcao]
    if obrigatoria:
        raise ValueError(f"Coluna não identificada. Colunas disponíveis: {df.columns.tolist()}")
    return None

def ler_conjunto(nome, df, pasta):
    col_id = escolher_coluna(df, COLUNA_ID, IDS)
    exige_rotulo = nome != "teste" or TESTE_ROTULADO is True
    col_y = (None if nome == "teste" and TESTE_ROTULADO is False else
             escolher_coluna(df, COLUNA_ROTULO, ROTULOS, exige_rotulo))
    if df.empty or df[col_id].isna().any():
        raise ValueError(f"{nome}: CSV vazio ou identificador ausente.")
    ids = df[col_id].str.strip().str.replace("\\", "/", regex=False)
    if (ids == "").any():
        raise ValueError(f"{nome}: identificador vazio.")
    # Indexação por nome e por stem para aceitar IDs com ou sem extensão.
    por_nome, por_id = {}, {}
    for img in pasta.rglob("*"):
        if img.is_file() and img.suffix.lower() in EXTENSOES:
            por_nome.setdefault(img.name.lower(), []).append(img)
            por_id.setdefault(img.stem.lower(), []).append(img)
    caminhos = []
    for identificador in ids:
        arquivo = PurePosixPath(identificador).name
        p = Path(arquivo)
        opcoes = (por_nome.get(arquivo.lower(), []) if p.suffix.lower() in EXTENSOES
                  else por_id.get(arquivo.lower(), []))
        if len(opcoes) != 1:
            raise ValueError(f"{nome}: imagem ausente ou ambígua: {identificador} ({len(opcoes)} opções)")
        caminhos.append(opcoes[0])
    tabela = pd.DataFrame({"id_code": [p.stem for p in caminhos],
                           "caminho": [str(p) for p in caminhos], "conjunto": nome})
    if tabela["id_code"].str.lower().duplicated().any():
        raise ValueError(f"{nome}: imagem repetida no CSV.")
    if col_y is not None:
        graus = pd.to_numeric(df[col_y], errors="coerce")
        if not graus.isin([0, 1, 2, 3, 4]).all():
            raise ValueError(f"{nome}: rótulos inválidos na coluna {col_y}. Esperados graus inteiros de 0 a 4. Se o teste tiver rótulos fictícios, configure TESTE_ROTULADO = False.")
        tabela["diagnosis"] = graus.astype("int32").to_numpy()
        tabela["rotulo_binario"] = (tabela["diagnosis"] >= 2).astype("int32")
    else:
        tabela["diagnosis"] = pd.Series(pd.NA, index=tabela.index, dtype="Int32")
        tabela["rotulo_binario"] = pd.Series(pd.NA, index=tabela.index, dtype="Int32")
    print(f"{nome}: {len(tabela)} imagens associadas; {sum(map(len, por_id.values()))} arquivos na pasta; ID={col_id}; rótulo={col_y}")
    return tabela

conjuntos = {nome: ler_conjunto(nome, TABELAS_BRUTAS[nome], pasta)
             for nome, (_, pasta) in CAMINHOS.items()}
treino, validacao, teste = (conjuntos[n] for n in DIVISOES)

# Não permite reutilização de identificadores entre as três divisões.
for a, b in [("treino", "validacao"), ("treino", "teste"), ("validacao", "teste")]:
    repetidos = set(conjuntos[a].id_code.str.lower()) & set(conjuntos[b].id_code.str.lower())
    if repetidos:
        raise ValueError(f"IDs repetidos entre {a} e {b}: {sorted(repetidos)[:10]}")
print("Nenhum identificador repetido entre os conjuntos.")
print("A checagem por ID não exclui duplicatas de conteúdo com nomes diferentes.")
amostras = treino  # Exemplos visuais usam apenas imagens de treino.
display(treino.head())

## 5. Distribuição dos dados

Mantemos os três conjuntos de origem e calculamos a distribuição real, sem impor as quantidades do artigo. A classe binária 0 reúne imagens sem retinopatia e com retinopatia leve. A classe 1 reúne os graus moderado, grave e proliferativo. A classe 0 não significa necessariamente ausência de retinopatia.

Se o teste não tiver rótulos, a tabela mostrará essa ausência, sem atribuir uma classe artificial.

In [ ]:
total = sum(len(df) for df in conjuntos.values())
resumo = pd.DataFrame([
    {"conjunto": nome, "imagens": len(df), "percentual": round(100*len(df)/total, 2),
     "rotuladas": int(df.diagnosis.notna().sum())}
    for nome, df in conjuntos.items()
])
display(resumo)
rotulados = pd.concat(conjuntos.values(), ignore_index=True).dropna(subset=["diagnosis"])
for coluna, classes in [("diagnosis", range(5)), ("rotulo_binario", range(2))]:
    tabela = pd.crosstab(rotulados[coluna], rotulados["conjunto"]).reindex(index=list(classes), fill_value=0)
    print("\nDistribuição:", coluna)
    display(tabela)
    tabela.plot(kind="bar", figsize=(9, 4), rot=0)
    plt.ylabel("Quantidade de imagens")
    plt.tight_layout()
    plt.show()

## 6. Visualizar imagens originais

Exibimos uma imagem de treino de cada grau disponível, seguindo a organização didática do notebook de referência `Tratamento_Imagens.ipynb`. As imagens serão carregadas do seu ZIP, não de exemplos gerados.

In [ ]:
nomes_classes = {
    0: "0. Sem RD", 1: "1. Leve", 2: "2. Moderada",
    3: "3. Grave", 4: "4. Proliferativa",
}

fig, axes = plt.subplots(1, 5, figsize=(17, 4))
for grau, ax in enumerate(axes):
    candidatas = amostras.loc[amostras["diagnosis"] == grau]
    if candidatas.empty:
        ax.text(0.5, 0.5, "Sem imagem\nnessa classe", ha="center", va="center")
    else:
        caminho = candidatas.iloc[0]["caminho"]
        with Image.open(caminho) as img:
            ax.imshow(img.convert("RGB"))
    ax.set_title(nomes_classes[grau])
    ax.axis("off")
plt.suptitle("Exemplos reais do APTOS 2019 antes do pré-processamento", y=1.05)
plt.tight_layout()
plt.show()

## 7. Redimensionamento e normalização

O redimensionamento direto reduz todas as imagens a 128 × 128 pixels, sem preservar necessariamente a proporção original. A conversão para RGB mantém três canais. A normalização converte os valores inteiros de 0 a 255 para números `float32` de 0 a 1 pela divisão por 255. O efeito visual é o mesmo; muda a escala numérica usada pela rede neural.

In [ ]:
TAMANHO = (128, 128)  # OpenCV usa (largura, altura)


def carregar_rgb(caminho):
    """Lê uma imagem real e devolve um array RGB uint8."""
    with Image.open(caminho) as imagem:
        return np.asarray(imagem.convert("RGB"))


def redimensionar_128(imagem_rgb):
    """Redimensionamento direto conforme a dimensão descrita no EdgeRetina."""
    h, w = imagem_rgb.shape[:2]
    interpolacao = cv2.INTER_AREA if (h > 128 or w > 128) else cv2.INTER_LINEAR
    return cv2.resize(imagem_rgb, TAMANHO, interpolation=interpolacao)


def normalizar_01(imagem_uint8):
    """Converte [0, 255] para float32 no intervalo [0, 1]."""
    return imagem_uint8.astype(np.float32) / np.float32(255.0)


def preprocessar_imagem(caminho):
    """Etapa determinística, idêntica para treino, validação e teste."""
    return normalizar_01(redimensionar_128(carregar_rgb(caminho)))

exemplo = amostras.iloc[0]
original = carregar_rgb(exemplo["caminho"])
reduzida = redimensionar_128(original)
normalizada = normalizar_01(reduzida)

fig, eixos = plt.subplots(1, 3, figsize=(13, 4))
eixos[0].imshow(original)
eixos[0].set_title(f"Original: {original.shape[1]} × {original.shape[0]}")
eixos[1].imshow(reduzida)
eixos[1].set_title("Redimensionada: 128 × 128")
eixos[2].imshow(normalizada)
eixos[2].set_title("Normalizada: valores entre 0 e 1")
for eixo in eixos:
    eixo.axis("off")
plt.tight_layout()
plt.show()

print("Imagem original:", original.shape, original.dtype)
print("Redimensionada:", reduzida.shape, reduzida.dtype)
print("Normalizada:", normalizada.shape, normalizada.dtype)
print("Mínimo, máximo:", float(normalizada.min()), float(normalizada.max()))

## 8. Conferência da normalização

As verificações abaixo conferem dimensões, tipo de dado, intervalo de pixels e correspondência entre a imagem redimensionada e a normalizada. O histograma compara os valores em uma escala comum.

In [ ]:
assert original.ndim == 3 and original.shape[2] == 3
assert reduzida.shape == (128, 128, 3)
assert reduzida.dtype == np.uint8
assert normalizada.shape == (128, 128, 3)
assert normalizada.dtype == np.float32
assert 0.0 <= float(normalizada.min()) <= float(normalizada.max()) <= 1.0
assert np.allclose(normalizada * 255.0, reduzida.astype(np.float32), atol=1e-4)

print("Verificações aprovadas.")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(reduzida.ravel(), bins=50, alpha=0.55, label="Imagem redimensionada, 0 a 255")
ax.hist((normalizada * 255.0).ravel(), bins=50, alpha=0.55,
        histtype="step", linewidth=2, label="Normalizada multiplicada por 255")
ax.set_xlabel("Valor do pixel na mesma escala para comparação")
ax.set_ylabel("Quantidade de valores")
ax.set_title("A normalização altera a escala numérica, não o contraste relativo")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Aumento de dados no treinamento

A inversão horizontal ocorre com probabilidade de 50%. O contraste usa um fator uniforme entre 0,90 e 1,10 em torno da média de cada canal. Os valores são limitados ao intervalo de 0 a 1. A figura mostra quatro transformações possíveis de uma mesma imagem de treino. Não alteramos os rótulos ou os conjuntos de validação e teste.

In [ ]:
def aumentar_imagem_numpy(imagem_01, gerador):
    """Exemplo visual do aumento. Entrada e saída: RGB float32 em [0, 1]."""
    saida = imagem_01.copy()
    inverteu = gerador.random() < 0.5
    if inverteu:
        saida = np.fliplr(saida).copy()

    fator = float(gerador.uniform(0.9, 1.1))
    media_por_canal = saida.mean(axis=(0, 1), keepdims=True)
    saida = np.clip((saida - media_por_canal) * fator + media_por_canal, 0.0, 1.0)
    return saida.astype(np.float32), inverteu, fator

rng = np.random.default_rng(SEMENTE)
fig, eixos = plt.subplots(1, 5, figsize=(17, 4))
eixos[0].imshow(normalizada)
eixos[0].set_title("Sem aumento")
eixos[0].axis("off")
for indice in range(1, 5):
    imagem_aumentada, invertida, fator_contraste = aumentar_imagem_numpy(normalizada, rng)
    eixos[indice].imshow(imagem_aumentada)
    eixos[indice].set_title(
        f"Exemplo {indice}\nFlip: {'sim' if invertida else 'não'}; contraste: {fator_contraste:.3f}"
    )
    eixos[indice].axis("off")
    assert imagem_aumentada.shape == (128, 128, 3)
    assert 0.0 <= float(imagem_aumentada.min()) <= float(imagem_aumentada.max()) <= 1.0
plt.tight_layout()
plt.show()

## 10. Preparar o pipeline completo em TensorFlow

Esta etapa cria lotes de 32 imagens sem carregar todo o dataset na memória de uma vez. O pipeline percorre todos os registros do ZIP, aplica as operações determinísticas nos três conjuntos e aplica o aumento de dados somente no treinamento, a cada passagem. Se o teste não for rotulado, `test_ds` terá somente imagens para inferência. Nenhum modelo é treinado aqui.

O TensorFlow costuma vir instalado no Colab. Se necessário, instale com `%pip install tensorflow` e reinicie a sessão.

In [ ]:
import tensorflow as tf

tf.random.set_seed(SEMENTE)
BATCH_SIZE = 32


def ler_para_tensorflow(valor):
    """Recebe um caminho como bytes e retorna um array normalizado."""
    if isinstance(valor, np.ndarray):
        valor = valor.item()
    if isinstance(valor, bytes):
        caminho = valor.decode("utf-8")
    else:
        caminho = str(valor)
    return preprocessar_imagem(caminho)


def decodificar_tf(caminho, rotulo):
    imagem = tf.numpy_function(ler_para_tensorflow, [caminho], Tout=tf.float32)
    imagem.set_shape((128, 128, 3))
    rotulo = tf.cast(rotulo, tf.float32)
    return imagem, rotulo


def aumentar_apenas_treino_tf(imagem, rotulo):
    imagem = tf.image.random_flip_left_right(imagem)
    imagem = tf.image.random_contrast(imagem, lower=0.9, upper=1.1)
    imagem = tf.clip_by_value(imagem, 0.0, 1.0)
    return imagem, rotulo


def decodificar_sem_rotulo(caminho):
    imagem = tf.numpy_function(ler_para_tensorflow, [caminho], Tout=tf.float32)
    imagem.set_shape((128, 128, 3))
    return imagem


def construir_dataset(df, treinamento=False):
    caminhos = df["caminho"].astype(str).to_numpy()
    rotulado = df["rotulo_binario"].notna().all()
    if rotulado:
        labels = df["rotulo_binario"].astype("float32").to_numpy()
        dataset = tf.data.Dataset.from_tensor_slices((caminhos, labels))
    else:
        if treinamento:
            raise ValueError("Treino precisa de rótulos.")
        dataset = tf.data.Dataset.from_tensor_slices(caminhos)
        return dataset.map(decodificar_sem_rotulo, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    if treinamento:
        dataset = dataset.shuffle(len(df), seed=SEMENTE, reshuffle_each_iteration=True)
    dataset = dataset.map(decodificar_tf, num_parallel_calls=tf.data.AUTOTUNE)
    if treinamento:
        dataset = dataset.map(aumentar_apenas_treino_tf, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = construir_dataset(treino, treinamento=True)
val_ds = construir_dataset(validacao, treinamento=False)
test_ds = construir_dataset(teste, treinamento=False)
print("Treino:", len(treino), "| Validação:", len(validacao), "| Teste:", len(teste))
print("Teste rotulado:", teste.rotulo_binario.notna().all())

imagens_lote, rotulos_lote = next(iter(train_ds))
print("Formato de um lote de treinamento:", imagens_lote.shape)
print("Formato dos rótulos:", rotulos_lote.shape)
print("Intervalo dos pixels:", float(tf.reduce_min(imagens_lote)), float(tf.reduce_max(imagens_lote)))
print("Tipos:", imagens_lote.dtype, rotulos_lote.dtype)

assert imagens_lote.shape[1:] == (128, 128, 3)
assert imagens_lote.dtype == tf.float32
assert float(tf.reduce_min(imagens_lote)) >= 0.0
assert float(tf.reduce_max(imagens_lote)) <= 1.0

## 11. Visualizar lotes de validação e teste

Os dois conjuntos passam somente por redimensionamento e normalização. A visualização do teste informa os rótulos apenas quando eles existem no CSV e não foram desativados na configuração.

In [ ]:
for nome, ds in [("Validação", val_ds), ("Teste", test_ds)]:
    lote = next(iter(ds))
    if isinstance(lote, tuple):
        imagens, labels = lote
    else:
        imagens, labels = lote, None
    assert imagens.shape[1:] == (128, 128, 3)
    assert float(tf.reduce_min(imagens)) >= 0 and float(tf.reduce_max(imagens)) <= 1
    fig, eixos = plt.subplots(2, 4, figsize=(12, 6))
    for i, ax in enumerate(eixos.flat):
        if i < len(imagens):
            ax.imshow(imagens[i].numpy())
            ax.set_title(f"Classe: {int(labels[i].numpy())}" if labels is not None else "Sem rótulo")
        ax.axis("off")
    plt.suptitle(f"{nome}: sem aumento de dados")
    plt.tight_layout()
    plt.show()

## 12. Exportar as imagens processadas (opcional)

Para salvar o dataset completo, altere `EXPORTAR = True`. Por padrão, o código exporta todas as imagens em PNG RGB de 128 × 128 e mantém os CSVs separados. Os PNGs guardam pixels inteiros de 0 a 255; a normalização para `float32` deve ser feita na leitura, como na função `preprocessar_imagem`. O aumento de dados é dinâmico e não é salvo, para preservar a divisão e evitar duplicação de arquivos. Os rótulos desconhecidos ficam vazios.

In [ ]:
EXPORTAR = False
MAXIMO_POR_DIVISAO = None  # Ex.: 30 para exportar apenas uma amostra de cada conjunto.

if EXPORTAR:
    SAIDA = Path(tempfile.mkdtemp(prefix="processado_", dir=TRABALHO))
    for nome, tabela in conjuntos.items():
        destino = SAIDA / nome
        destino.mkdir()
        subset = tabela if MAXIMO_POR_DIVISAO is None else tabela.head(MAXIMO_POR_DIVISAO)
        linhas = []
        for linha in subset.itertuples(index=False):
            rgb128 = redimensionar_128(carregar_rgb(linha.caminho))
            arquivo = f"{linha.id_code}.png"
            Image.fromarray(rgb128).save(destino / arquivo)
            linhas.append({"id_code": linha.id_code, "diagnosis": linha.diagnosis,
                           "rotulo_binario": linha.rotulo_binario, "arquivo": f"{nome}/{arquivo}"})
        pd.DataFrame(linhas).to_csv(SAIDA / f"{nome}.csv", index=False)
        print(f"{nome}: {len(linhas)} imagens exportadas")
    caminho_zip = shutil.make_archive(str(TRABALHO / "APTOS_EdgeRetina_128"), "zip", SAIDA)
    print("ZIP criado:", caminho_zip)
else:
    print("Exportação desativada. Altere EXPORTAR para True se desejar salvar as imagens.")

In [ ]:
if EXPORTAR:
    from google.colab import files
    files.download(caminho_zip)
else:
    print("Nenhuma exportação solicitada.")

## 13. Conclusão e limites

O notebook conserva a divisão fornecida no ZIP, verifica a correspondência entre CSVs e imagens, calcula a distribuição das classes, demonstra as transformações e prepara três pipelines. As operações principais seguem o EdgeRetina. A divisão e alguns parâmetros de implementação são adaptações, não uma reprodução exata dos experimentos publicados. Não há resultados de acurácia, sensibilidade ou AUC, pois não treinamos um modelo.

Os nomes internos das colunas do seu ZIP precisam ser confirmados durante a execução. Se a leitura falhar, consulte as colunas exibidas na seção 4. O teste só pode ser usado para medir desempenho se seus rótulos forem reais e confiáveis. A checagem de IDs não garante separação por paciente ou ausência de duplicatas de conteúdo. Uso acadêmico, não destinado a diagnóstico clínico.

### Referências

- Guidoum, A. et al. *EdgeRetina: Hybrid multimedia architecture for diabetic retinopathy screening on low-cost mobiles*. IAES International Journal of Robotics and Automation, v. 15, n. 1, 2026, p. 234–246. https://doi.org/10.11591/ijra.v15i1.pp234-246.
- APTOS. *APTOS 2019 Blindness Detection*. https://www.kaggle.com/competitions/aptos2019-blindness-detection/data.
- OIO-MAIS. Notebook `Tratamento_Imagens.ipynb`, referência para a organização das demonstrações. https://github.com/ChrisPRodrigues/OIO-MAIS/tree/main.
- Google Colab, acesso a arquivos: https://colab.research.google.com/notebooks/io.ipynb.